<div class="dark-title" style="background:linear-gradient(90deg,#9a3412,#7c2d12); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

# 📁 Lesson 2 · 本节精华

</div>

<small>对应主课:[2_a_🔥_files.ipynb](./2_a_🔥_files.ipynb)</small>

## 🎯 一句话定位

> 给 agent 配 **虚拟文件系统** —— 把 token-heavy 的信息(网页原文、长输出)**搬出主对话**,需要时再按需取回。
>
> 🧹 **本质 = 用文件做"外脑",保持主 context 又小又聚焦**。

## 📚 你应该学到的 8 件事

| # | 概念 | 你应该记住的一句话 |
|---|---|---|
| 1 | 🧹 **Context Offloading** | 把长信息存到文件,**主对话只留摘要** —— Manus、HF、Anthropic 都这么做 |
| 2 | 🗄️ **虚拟文件系统** | 本质是 `state.files: dict[str, str]`,key=路径,value=内容 |
| 3 | 🛠️ **三件套** | `ls()` 列文件,`read_file()` 读(带分页),`write_file()` 写 |
| 4 | 🔄 **`file_reducer`** | 浅合并 `{**left, **right}`,**并行写 / 子 agent 提交必需** |
| 5 | 📑 **`description=` vs docstring** | `description=` 给 LLM 看,docstring 给程序员看(详见 [2_b](./2_b_⭐️_description=_VS_docstring_for_tools.ipynb)) |
| 6 | 📖 **`offset` + `limit`** | 分页读,**对抗大文件爆 context**(详见 [2_c](./2_c_⭐️_write_file_分页读取.ipynb)) |
| 7 | ⚠️ **错误信息写给 LLM** | 错误是给机器读的,要**具体、技术化**,不是 "哎呀出错了 😅" |
| 8 | 🎬 **Save → Search → Read-Back** | 完整执行轨迹(详见 [2_d](./2_d_⭐️_messages_完整执行轨迹.ipynb)) |

## 🧠 核心心智模型:**文件 = 注意力开关**

本节最反直觉、但最重要的一个洞察:

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

> 💡 **虚拟文件系统的真正作用,不是"存数据",而是"控制 LLM 注意力"**。
>
> - 想让 LLM **暂时忘记**某段长内容 → `write_file` 把它移出 messages,只留文件名(摘要)
>
> - 想让 LLM **想起**它 → `read_file` 把它塞回 messages 末尾

</div>

回忆 [2_d](./2_d_⭐️_messages_完整执行轨迹.ipynb) 那个 MCP overview 例子,LLM 的标准动作 = **Save → Search → Read-Back**:

```
  ┌─ Save 计划:  write_file("mcp_request.txt", "Topics to cover: ...")
  │
  ├─ Search:    web_search("MCP overview")
  │              ↓ 大段搜索结果挤进 messages
  │
  ├─ Read-Back: read_file("mcp_request.txt")
  │              ↓ 计划又回到 messages 末尾
  │
  └─ Answer:    按照计划一项一项答
```

**没有 read-back,LLM 容易跑题**(被搜索结果带偏)。**有了 read-back,计划永远在视线内**。

## 💎 3 个值得记住的工程精髓

### 1️⃣ `description=` 接管 LLM 视角

```python
@tool(description=LS_DESCRIPTION)         # ← 🤖 给 LLM 看的(几十行长 prompt)
def ls(state):
    """List all files."""                 # ← 👨‍💻 给程序员看的(一句话)
```

**一旦写了 `description=`,docstring 对 LLM 就消失**(除非用 `parse_docstring=True` 暴露 `Args:` 段)。详见 [2_b](./2_b_⭐️_description=_VS_docstring_for_tools.ipynb)。

### 2️⃣ `read_file` 用 `offset/limit` 分页

```python
read_file("foo.py")               # 前 2000 行
read_file("foo.py", offset=2000)  # 第 2001~4000 行
```

**让 LLM 自己决定读多少**,避免一口气吞掉 50000 行爆 context。**这是 Claude Code `Read` 工具的同款设计**。

### 3️⃣ 错误信息 = 给 LLM 的"机器日志"

```python
return f"Error: File '{file_path}' not found"   # ✅ LLM 看得懂、能据此重试
```

不要写 "哎呀,文件没找到 😢 请重试" —— **目标读者是 LLM,不是终端用户**。具体、技术化、信息密集。

## 🔄 关于 `file_reducer` 的一个**容易踩的坑**

本节定义了 `file_reducer`(合并 dict),但**本节示例其实不需要它** —— 因为 `write_file` 自己已经"读旧 + 加新 + 返回完整 dict":

```python
def write_file(file_path, content, state, ...):
    files = state.get("files", {})       # ① 读出全部已有 files
    files[file_path] = content            # ② 在已有基础上加
    return Command(update={"files": files})  # ③ 返回完整 dict
```

**默认 reducer(覆盖)与 `file_reducer`(合并)结果一样** —— 因为 update 本身已经是"完整版"。

**那 `file_reducer` 真正必要的场景是什么?**

| 场景 | 必要性 |
|---|---|
| 单线程顺序写 | ❌ 不必要(工具自己合并了)|
| **并行写**(同一 AIMessage 多 tool_call)| ✅ **必需** —— 各 call 读到同一快照,不合并就丢 |
| **子 agent 提交 files 回父**(Lesson 3 用到)| ✅ **必需** —— 父 files + 子 files 必须合并 |

> 🔑 **生产级 agent 几乎总有并行 + 子 agent → 几乎总需要 `file_reducer`**。本节定义它,是为后面 Lesson 3-4 铺路。

## 🌍 真实产品对应 + 一句话

| 你学的 | 现实里的实现 |
|---|---|
| 🧹 Context offloading | **Manus / HF Open Deep Research / Anthropic Multi-Agent** 都用 |
| 📖 `offset+limit` 分页 | **Claude Code 的 `Read` 工具**完全同款 |
| ✍️ 写文件作为长期记忆 | **Devin / Cursor Composer** 的 sandbox workspace |
| 🚫 默认禁用扩展 / 强约束 LLM | 错误信息只给 LLM 看 → 不暴露给用户(产品边界) |

## ✨ 一句话带走

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

> 🔑 **文件系统是 LLM 的"外脑 + 注意力开关"**。
>
> - 想忘 → 写文件,主对话只留文件名
>
> - 想起 → 读文件,内容回到 messages 末尾
>
> **deep agent 之所以能跑 50+ 工具调用而不崩,关键就在于把 token 大户搬到文件里**,而不是堆在 prompt 里。

</div>